This is the scratchpad to work out the code before rewriting it as a block script in volatility_renewable_share. I find it easier to debug and create in a notebook.

In [40]:
from pathlib import Path
import sys
sys.path.append("..") #notebook workaround for relative imports - not needed in the final script file
import pandas as pd
import statsmodels.formula.api as smf
from src.pipeline_utils import get_engine

In [41]:
engine = get_engine()
with engine.begin() as conn:
    df_generation = pd.read_sql("SELECT * FROM generation", conn)

In [42]:
df_generation.head()

,zone,timestamp,production_type,value_mw
0,DE_LU,2026-09-14 22:00:00+00:00,Biomass,3847.93709
1,DE_LU,2026-09-14 22:15:00+00:00,Biomass,3832.25409
2,DE_LU,2026-09-14 22:30:00+00:00,Biomass,3821.68579
3,DE_LU,2026-09-14 22:45:00+00:00,Biomass,3812.94209
4,DE_LU,2026-09-14 23:00:00+00:00,Biomass,3798.16749


The where here is for future proofing. The one below. Build into the function for the script.

In [43]:
with engine.begin() as conn:
    df_price = pd.read_sql("SELECT * FROM prices WHERE zone IN ('DE_LU', 'SE_4')", conn)

In [44]:
df_price.head()

,zone,timestamp,price_eur_mwh
0,DE_LU,2025-12-01 23:00:00+00:00,74.08
1,DE_LU,2025-12-01 23:15:00+00:00,69.92
2,DE_LU,2025-12-01 23:30:00+00:00,63.16
3,DE_LU,2025-12-01 23:45:00+00:00,55.95
4,DE_LU,2025-12-02 00:00:00+00:00,65.41


The clip here in the code below is to kill hydro water storage when negative which isnt strictly generation.

In [45]:
generation_wide_df = df_generation.pivot(index=['zone','timestamp'], columns='production_type', values='value_mw').clip(lower=0)

In [46]:
generation_wide_df.head()

production_type                      Biomass  Fossil Brown coal/Lignite  \
zone  timestamp                                                           
DE_LU 2025-12-01 23:00:00+00:00  3747.499144                 6912.68041   
      2025-12-01 23:15:00+00:00  3746.302984                 6923.59182   
      2025-12-01 23:30:00+00:00  3757.497088                 6915.54587   
      2025-12-01 23:45:00+00:00  3764.604516                 6854.23653   
      2025-12-02 00:00:00+00:00  3755.147836                 6596.57927   

production_type                  Fossil Coal-derived gas  Fossil Gas  \
zone  timestamp                                                        
DE_LU 2025-12-01 23:00:00+00:00                492.45667  6304.37520   
      2025-12-01 23:15:00+00:00                574.95667  6106.24293   
      2025-12-01 23:30:00+00:00                580.10334  6013.96636   
      2025-12-01 23:45:00+00:00                591.70667  5972.12206   
      2025-12-02 00:00:00+00:00                605.11334  6006.44944   

production_type                  Fossil Hard coal  Fossil Oil  Geothermal  \
zone  timestamp                                                             
DE_LU 2025-12-01 23:00:00+00:00        2659.07593   401.29800    34.06288   
      2025-12-01 23:15:00+00:00        2569.16804   399.94200    34.23812   
      2025-12-01 23:30:00+00:00        2499.61620   399.94200    34.14856   
      2025-12-01 23:45:00+00:00        2500.13420   399.94200    34.38876   
      2025-12-02 00:00:00+00:00        2497.30430   400.20867    34.43824   

production_type                  Hydro Pumped Storage  \
zone  timestamp                                         
DE_LU 2025-12-01 23:00:00+00:00                   0.0   
      2025-12-01 23:15:00+00:00                   0.0   
      2025-12-01 23:30:00+00:00                   0.0   
      2025-12-01 23:45:00+00:00                   0.0   
      2025-12-02 00:00:00+00:00                   0.0   

production_type                  Hydro Run-of-river and poundage  \
zone  timestamp                                                    
DE_LU 2025-12-01 23:00:00+00:00                       1447.43006   
      2025-12-01 23:15:00+00:00                       1449.20300   
      2025-12-01 23:30:00+00:00                       1446.75980   
      2025-12-01 23:45:00+00:00                       1439.94270   
      2025-12-02 00:00:00+00:00                       1442.12410   

production_type                  Hydro Water Reservoir      Other  \
zone  timestamp                                                     
DE_LU 2025-12-01 23:00:00+00:00                24.1717  222.39623   
      2025-12-01 23:15:00+00:00                31.3956  224.18423   
      2025-12-01 23:30:00+00:00                20.2129  224.41023   
      2025-12-01 23:45:00+00:00                23.1088  224.39323   
      2025-12-02 00:00:00+00:00                24.7121  218.17923   

production_type                  Other renewable     Solar    Waste  \
zone  timestamp                                                       
DE_LU 2025-12-01 23:00:00+00:00        89.339876  8.204348  735.495   
      2025-12-01 23:15:00+00:00        89.642020  8.257364  708.342   
      2025-12-01 23:30:00+00:00        89.952904  7.904608  708.654   
      2025-12-01 23:45:00+00:00        89.879208  7.934012  713.286   
      2025-12-02 00:00:00+00:00        89.782412  7.286076  706.126   

production_type                  Wind Offshore  Wind Onshore  
zone  timestamp                                               
DE_LU 2025-12-01 23:00:00+00:00       5419.125  29166.695404  
      2025-12-01 23:15:00+00:00       5644.271  28831.382340  
      2025-12-01 23:30:00+00:00       5711.357  28613.092408  
      2025-12-01 23:45:00+00:00       5761.216  28311.670976  
      2025-12-02 00:00:00+00:00       5819.805  28224.392636

The following works but should be prettier and can be a function. At the end there we do rolling average of generation share.

In [ ]:
genx=generation_wide_df.reset_index().copy()
genx['renewables_generation']=genx[['Wind Onshore','Wind Offshore','Solar']].sum(axis=1)
production_cols = generation_wide_df.columns.tolist()
genx['total_generation_mw']=genx[production_cols].sum(axis=1)
genx['renewables_share']=genx['renewables_generation']/genx['total_generation_mw']
renewables_share_df=genx[["zone", "timestamp", "renewables_share"]].copy()
renewables_share_df["renewables_share_24h"] = (renewables_share_df.groupby("zone")["renewables_share"].rolling(96).mean().reset_index(level=0, drop=True))


In [48]:
renewables_share_df.head()

production_type,zone,timestamp,renewables_share
0,DE_LU,2025-12-01 23:00:00+00:00,0.599921
1,DE_LU,2025-12-01 23:15:00+00:00,0.601382
2,DE_LU,2025-12-01 23:30:00+00:00,0.602077
3,DE_LU,2025-12-01 23:45:00+00:00,0.601194
4,DE_LU,2025-12-02 00:00:00+00:00,0.603454


Ok thats one of our final dataframes ready, lets do the other one. The volatility one.

In [49]:
df_price.head()

,zone,timestamp,price_eur_mwh
0,DE_LU,2025-12-01 23:00:00+00:00,74.08
1,DE_LU,2025-12-01 23:15:00+00:00,69.92
2,DE_LU,2025-12-01 23:30:00+00:00,63.16
3,DE_LU,2025-12-01 23:45:00+00:00,55.95
4,DE_LU,2025-12-02 00:00:00+00:00,65.41


In [56]:
df_generation["timestamp"] = pd.to_datetime(df_generation["timestamp"])
renewables_share_df["timestamp"] = pd.to_datetime(renewables_share_df["timestamp"])
df_price["timestamp"] = pd.to_datetime(df_price["timestamp"])
df_price = df_price.sort_values(["zone", "timestamp"]) #NEEDS TO BE SORTED!

In [57]:
df_price["volatility"] = (df_price.groupby("zone")["price_eur_mwh"].rolling(96).std().reset_index(level=0, drop=True))

In [58]:
df_price

,zone,timestamp,price_eur_mwh,volatility
0,DE_LU,2025-12-01 23:00:00+00:00,74.08,NaN
1,DE_LU,2025-12-01 23:15:00+00:00,69.92,NaN
2,DE_LU,2025-12-01 23:30:00+00:00,63.16,NaN
3,DE_LU,2025-12-01 23:45:00+00:00,55.95,NaN
4,DE_LU,2025-12-02 00:00:00+00:00,65.41,NaN
...,...,...,...,...
57013,SE_4,2026-09-24 21:00:00+00:00,192.00,47.794974
57014,SE_4,2026-09-24 21:15:00+00:00,183.54,47.881856
57015,SE_4,2026-09-24 21:30:00+00:00,177.23,47.920801
57016,SE_4,2026-09-24 21:45:00+00:00,174.83,47.927648


And time for the actual analysis file:

In [59]:
analysis_df = df_price.merge(renewables_share_df, on=["zone", "timestamp"], validate="one_to_one")

In [61]:
analysis_df.head(10)

,zone,timestamp,price_eur_mwh,volatility,renewables_share
0,DE_LU,2025-12-01 23:00:00+00:00,74.08,NaN,0.599921
1,DE_LU,2025-12-01 23:15:00+00:00,69.92,NaN,0.601382
2,DE_LU,2025-12-01 23:30:00+00:00,63.16,NaN,0.602077
3,DE_LU,2025-12-01 23:45:00+00:00,55.95,NaN,0.601194
4,DE_LU,2025-12-02 00:00:00+00:00,65.41,NaN,0.603454
5,DE_LU,2025-12-02 00:15:00+00:00,58.49,NaN,0.604172
6,DE_LU,2025-12-02 00:30:00+00:00,56.59,NaN,0.602368
7,DE_LU,2025-12-02 00:45:00+00:00,54.79,NaN,0.601558
8,DE_LU,2025-12-02 01:00:00+00:00,55.08,NaN,0.606158
9,DE_LU,2025-12-02 01:15:00+00:00,54.24,NaN,0.606933


sanity check:

In [62]:
analysis_df.groupby("zone").size()

zone
DE_LU    28508
SE_4     28508
dtype: int64